# One-off migration: `usage_logs` -> batched schema

Converts each `usage_logs/{userId}` document from the old
`data: { <eventId>: {ts, type, package} }` map into the new batched shape:

```
data: { "<last-updated-ms>": [ {ts, type, package}, ... ] }
id:   "<userId>"
time: "<ISO8601 string>"
```

All existing events are placed under a single batch keyed by the document's
last-updated time (ms). Run top-to-bottom; keep `DRY_RUN = True` for a preview
before committing writes. Temporary/throwaway file.

In [ ]:
%pip install firebase-admin

In [ ]:
import json
from datetime import datetime, timezone

import firebase_admin
from firebase_admin import credentials, firestore

SERVICE_ACCOUNT_KEY = "service-account-key.json"
COLLECTION = "usage_logs"

if not firebase_admin._apps:
    firebase_admin.initialize_app(credentials.Certificate(SERVICE_ACCOUNT_KEY))
db = firestore.client()
print("Connected to project:", firebase_admin.get_app().project_id)

In [ ]:
def approx_bytes(obj) -> int:
    return len(json.dumps(obj, separators=(",", ":"), default=str).encode("utf-8"))


def last_updated_ms(doc: dict) -> int:
    """Batch key = the document's last-updated time in ms, best-effort."""
    t = doc.get("time")
    if isinstance(t, (int, float)):
        return int(t)
    if isinstance(t, str):
        try:
            return int(datetime.fromisoformat(t).timestamp() * 1000)
        except ValueError:
            pass
    return int(datetime.now(timezone.utc).timestamp() * 1000)


def is_already_migrated(data) -> bool:
    """New shape has list values (batches); old shape has map values (events)."""
    if not isinstance(data, dict) or not data:
        return False
    return all(isinstance(v, list) for v in data.values())


def to_batched(data: dict, batch_key: str) -> dict:
    """Old map {eventId: {ts,type,package}} -> {batch_key: [ {ts,type,package} ]}."""
    events = [v for v in data.values() if isinstance(v, dict)]
    return {batch_key: events}

In [ ]:
DRY_RUN = True  # set to False to actually write

for doc in db.collection(COLLECTION).stream():
    d = doc.to_dict() or {}
    data = d.get("data")

    if not isinstance(data, dict) or not data:
        print(f"[skip] {doc.id}: no data map")
        continue
    if is_already_migrated(data):
        print(f"[skip] {doc.id}: already migrated ({len(data)} batches)")
        continue

    batch_key = str(last_updated_ms(d))
    new_data = to_batched(data, batch_key)
    event_count = len(new_data[batch_key])
    time_str = d.get("time") or datetime.now(timezone.utc).isoformat()

    print(
        f"[{'dry' if DRY_RUN else 'write'}] {doc.id}: "
        f"{len(data)} events -> 1 batch[{batch_key}] of {event_count}; "
        f"~{approx_bytes(data)} -> ~{approx_bytes(new_data)} bytes"
    )

    if not DRY_RUN:
        doc.reference.update({"data": new_data, "time": time_str})

print("Done.")

## Optional: migrate the `history` sub-collection

Archived docs under `usage_logs/{userId}/history/{time}` share the old schema.
Run this only if you want them in the new shape too. Same `DRY_RUN` guard.

In [ ]:
DRY_RUN = True  # set to False to actually write

for parent in db.collection(COLLECTION).stream():
    for doc in parent.reference.collection("history").stream():
        d = doc.to_dict() or {}
        data = d.get("data")
        if not isinstance(data, dict) or not data or is_already_migrated(data):
            print(f"[skip] {parent.id}/history/{doc.id}")
            continue

        batch_key = str(last_updated_ms(d))
        new_data = to_batched(data, batch_key)
        time_str = d.get("time") or datetime.now(timezone.utc).isoformat()

        print(
            f"[{'dry' if DRY_RUN else 'write'}] {parent.id}/history/{doc.id}: "
            f"{len(data)} events -> 1 batch of {len(new_data[batch_key])}"
        )
        if not DRY_RUN:
            doc.reference.update({"data": new_data, "time": time_str})

print("Done.")